# Segmentasi Semantik dengan U-Net: Oxford-IIIT Pet

**Buku:** *Deep Learning dengan Python* — Bab 11, Segmentasi dan Deteksi Objek

Setiap piksel diklasifikasikan menjadi *hewan*, *latar*, atau *tepi*. Arsitektur U-Net menggabungkan konteks dari jalur *encoder* dengan detail spasial lewat *skip connection*.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Notebook ini memakai backend **TensorFlow** (dibutuhkan oleh `tf.data`/`tf.GradientTape`/ekspor).

**Perkiraan waktu (GPU T4):** ±20 menit

**Isi notebook:**
- Pipeline `tf.data` untuk pasangan gambar–masker, dengan augmentasi yang identik pada keduanya.
- U-Net: *encoder* (konvolusi + *pooling*), *decoder* (`Conv2DTranspose`), dan *skip connection*.
- Metrik segmentasi: akurasi piksel, IoU per kelas, mIoU, dan koefisien Dice.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf

## 1. Unduh data (±800 MB)

In [ ]:
import tarfile, pathlib, numpy as np
import matplotlib.pyplot as plt

for nama in ["images.tar.gz", "annotations.tar.gz"]:
    jalur = keras.utils.get_file(nama, origin=f"https://www.robots.ox.ac.uk/~vgg/data/pets/data/{nama}")
    if not pathlib.Path("data/pets/" + nama.split(".")[0]).exists():
        with tarfile.open(jalur) as berkas_tar:
            berkas_tar.extractall("data/pets")

jalur_gambar, jalur_masker = [], []
for p in sorted(pathlib.Path("data/pets/images").glob("*.jpg")):
    m = pathlib.Path("data/pets/annotations/trimaps") / (p.stem + ".png")
    if m.exists():
        jalur_gambar.append(str(p)); jalur_masker.append(str(m))
print(len(jalur_gambar), "pasangan gambar-masker")

KELAS = ["hewan", "latar", "tepi"]       # nilai trimap 1, 2, 3 -> 0, 1, 2

## 2. Pipeline data

In [ ]:
UKURAN = (160, 160)

def muat(jalur_g, jalur_m):
    g = tf.io.decode_image(tf.io.read_file(jalur_g), channels=3, expand_animations=False)
    g = tf.cast(tf.image.resize(g, UKURAN), "uint8")       # uint8 agar cache hemat RAM
    m = tf.io.decode_png(tf.io.read_file(jalur_m), channels=1)
    m = tf.image.resize(m, UKURAN, method="nearest")      # 'nearest' agar label tidak tercampur
    return g, tf.cast(m, "int32") - 1

def augmentasi(g, m):
    gabung = tf.concat([g, tf.cast(m, g.dtype)], axis=-1)  # balik gambar DAN masker bersamaan
    gabung = tf.image.random_flip_left_right(gabung)
    return gabung[..., :3], tf.cast(gabung[..., 3:], "int32")

rng = np.random.default_rng(42)
urutan = rng.permutation(len(jalur_gambar))
jalur_gambar = [jalur_gambar[i] for i in urutan]
jalur_masker = [jalur_masker[i] for i in urutan]
N_VAL, N_UJI = 1000, 500

def buat_ds(g, m, latih=False):
    ds = tf.data.Dataset.from_tensor_slices((g, m)).map(muat, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if latih:
        ds = ds.shuffle(1000).map(augmentasi, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

ds_latih = buat_ds(jalur_gambar[N_VAL + N_UJI:], jalur_masker[N_VAL + N_UJI:], latih=True)
ds_val = buat_ds(jalur_gambar[:N_VAL], jalur_masker[:N_VAL])
ds_uji = buat_ds(jalur_gambar[N_VAL:N_VAL + N_UJI], jalur_masker[N_VAL:N_VAL + N_UJI])

g, m = next(iter(ds_val))
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    ax[0, i].imshow(g[i].numpy().astype("uint8")); ax[1, i].imshow(m[i, :, :, 0], vmin=0, vmax=2)
    ax[0, i].axis("off"); ax[1, i].axis("off")
plt.show()

## 3. U-Net

In [ ]:
from keras import layers

def blok_konv(x, filter):
    for _ in range(2):
        x = layers.Conv2D(filter, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    return x

def unet(n_kelas=3, filter=(32, 64, 128, 256)):
    masukan = keras.Input(shape=UKURAN + (3,))
    x = layers.Rescaling(1 / 255)(masukan)
    lewatan = []
    for f in filter:                                   # encoder: resolusi turun, konteks naik
        x = blok_konv(x, f)
        lewatan.append(x)
        x = layers.MaxPooling2D(2)(x)
    x = blok_konv(x, filter[-1] * 2)                   # bottleneck 10x10
    for f, s in zip(reversed(filter), reversed(lewatan)):   # decoder: kembalikan resolusi
        x = layers.Conv2DTranspose(f, 2, strides=2, padding="same")(x)
        x = layers.Concatenate()([x, s])               # skip connection: detail spasial
        x = blok_konv(x, f)
    return keras.Model(masukan, layers.Conv2D(n_kelas, 1, activation="softmax")(x))

model = unet()
model.summary()

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy", keras.metrics.MeanIoU(num_classes=3, sparse_y_pred=False, name="miou")])
riwayat = model.fit(ds_latih, validation_data=ds_val, epochs=20,
                    callbacks=[keras.callbacks.ModelCheckpoint("unet_pets.keras", save_best_only=True,
                                                               monitor="val_miou", mode="max")])

## 4. Evaluasi: IoU dan Dice per kelas
Dihitung dari matriks konfusi tingkat piksel di seluruh data uji.

In [ ]:
model = keras.models.load_model("unet_pets.keras")
konfusi = np.zeros((3, 3), dtype=np.int64)
for g, m in ds_uji:
    pred = model.predict(g, verbose=0).argmax(-1).ravel()
    konfusi += np.bincount(3 * m.numpy().ravel() + pred, minlength=9).reshape(3, 3)

tp = np.diag(konfusi)
iou = tp / (konfusi.sum(0) + konfusi.sum(1) - tp)
dice = 2 * tp / (konfusi.sum(0) + konfusi.sum(1))
for k, i, d in zip(KELAS, iou, dice):
    print(f"{k:6s} IoU = {i:.3f} | Dice = {d:.3f}")
print(f"mIoU = {iou.mean():.3f} | akurasi piksel = {tp.sum() / konfusi.sum():.3f}")

In [ ]:
g, m = next(iter(ds_uji))
pred = model.predict(g, verbose=0).argmax(-1)
fig, ax = plt.subplots(3, 5, figsize=(14, 8))
for i in range(5):
    ax[0, i].imshow(g[i].numpy().astype("uint8"))
    ax[1, i].imshow(m[i, :, :, 0], vmin=0, vmax=2)
    ax[2, i].imshow(pred[i], vmin=0, vmax=2)
for a, t in zip(ax[:, 0], ["Gambar", "Masker asli", "Prediksi"]):
    a.set_ylabel(t, fontsize=12)
for a in ax.flat:
    a.set_xticks([]); a.set_yticks([])
plt.show()

Kelas *tepi* hampir selalu memiliki IoU terendah karena tipis dan ambigu. Inilah alasan akurasi piksel saja menyesatkan untuk segmentasi: kelas *latar* yang dominan membuatnya terlihat tinggi.